# Lab 3: Compiling Models for Dataflow Execution

<!-- The badge resolves against the PUBLIC mirror repo, which
     .github/workflows/sync-gpu-labs-student.yml keeps in step with
     gpu-labs/student/. Viewed inside the private instructor repo the image
     renders but the click-through 404s until that mirror has run. -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/gpu-labs/student/Lab3_Compiling_for_Dataflow_STUDENT.ipynb)

**Course**: *Dataflow Architectures for AI: From Principles to Practice*  
**Week**: 3, Compilation for Dataflow  
**Instructor**: Dr. Kaoutar El Maghraoui (IBM Research)  
**Lab Instructor**: Dr. Rashed Z. Bhatti (IBM Research)  
**Runtime**: Google Colab, GPU runtime required (T4)  
**Expected time**: 90–120 minutes

---

## Learning objectives
By the end of this lab you will be able to:
1. Compile a model with torch.compile and measure the speedup over eager mode.
2. Ablate the eager / aot_eager / inductor backends to locate where the speedup comes from.
3. Detect, count, and fix graph breaks that prevent whole-graph compilation.
4. Read the generated Triton vs cuBLAS kernels and relate fusion to dataflow.

## Prerequisites
Labs 1-2 complete, plus the Week 3 lecture on the compilation pipeline, graph breaks, and fusion.

## How this Self-Assessment Lab works

**Practice Lab**: You run this lab on **Google Colab** (free T4).  
**OPTIONAL SELF GRADING:** **A separate grader notebook is available on Coursera Labs for self-evaluation**.

| | Where | What happens |
|---|---|---|
| **1. This notebook** | Google Colab (T4 GPU) | Complete every `TODO`, make every **Checkpoint** pass, then run **Part 7** to write **`lab3_results.json`** |
| **2. Grader notebook** | Coursera Labs (no GPU) | For automated self assessment, paste the contents of `lab3_results.json`, then `Run All`. **40 ungraded points** |

The grader notebook has no GPU and no network, so it cannot re-compile your
model. Instead it checks that your numbers are **internally consistent** — each
speedup must follow from the two latencies it came from, the compilation
overhead from the first-call and steady-state timings, the subgraph count from
the graph-break count. Those relations hold for any real run on any GPU, which
is why a free-tier T4 earns a full score.

> **This lab is checked on consistency, never on speed.** `torch.compile` does
> **not** reliably beat eager on a block this small — the reference T4 run
> measured a speedup of just **1.05x**, and on some torch versions the
> `inductor` backend comes out *slower* than the `eager` backend. Both are real
> and instructive results, not failures. Your own numbers will differ with
> torch version and GPU, and **no check anywhere requires compilation to be
> faster.**

> **Students are encouraged to experiment with other GPUs and accelerators to expand their learning.** The free-tier T4 GPU is freely available on Google Colab. Re-running on an L4, V100 or A100 is genuinely instructive and you
> are encouraged to try it.

**Self-Assessment Lab ungraded points:**

| Part | Points | Comes from |
|---|---:|---|
| A — Structural invariants | 12 | Parts 1–6 of this notebook |
| B — Self-consistency | 18 | Parts 1–4 |
| C — Physics & plausibility | 10 | Parts 1–4 |
| **Ungraded notebook total** | **40** | fully auto-graded from your JSON for Self-Assessment |

**Graded Quiz points:**  
A 10 point graded Coursera Assignment quiz is given to assess your understanding from this lab experience.

| Quiz | Points | Comes from |
|---|---:|---|
| Concept Check quiz | 10 | Auto-graded Coursera Assignment |

The **Checkpoint** cells in this notebook are self-checks: they catch mistakes
as you work and assign no grade. Treat them as a preview — if a Checkpoint
fails here, the corresponding grader cell will fail there too.

### Working through this lab
1. Set **Runtime → Change runtime type → T4 GPU** before you run anything.
2. Run each cell in order. Make sure every **Checkpoint** cell passes (no `AssertionError`).
3. Fill in every `TODO`.
4. Run **Part 7** to produce **`lab3_results.json`**, then open the grader
   notebook on Coursera and paste it in for self-assessment.
5. Take the **Concept Check quiz** — a separate auto-graded Coursera
   Assignment worth **10 graded points**. It is mandatory, and you need 80% to
   pass.


## Environment Preflight

This cell detects your runtime environment and sets the execution mode:
- **Full GPU**: all exercises run on CUDA (recommended: T4 or better in Colab)
- **Reduced CPU**: GPU-dependent cells use CPU fallback with reduced workloads
- **Static read-only**: no execution; pre-computed outputs are displayed

In [ ]:
# === Environment Preflight ===
import sys
import importlib

# Check Python version
assert sys.version_info >= (3, 9), f"Python >= 3.9 required, got {sys.version}"

# Check PyTorch
try:
    import torch
    _torch_version = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
    assert _torch_version >= (2, 2), f"PyTorch >= 2.2 required, got {torch.__version__}"
except ImportError:
    raise RuntimeError("PyTorch not installed. Run: pip install torch>=2.2")

# Detect execution mode
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    MODE = "full-gpu"
    _device_name = torch.cuda.get_device_name(0)
else:
    DEVICE = torch.device("cpu")
    MODE = "reduced-cpu"
    _device_name = "CPU only"

print(f"╔══════════════════════════════════════════════════╗")
print(f"║  Environment Preflight                           ║")
print(f"╠══════════════════════════════════════════════════╣")
print(f"║  Python:    {sys.version.split()[0]:<37s}║")
print(f"║  PyTorch:   {torch.__version__:<37s}║")
print(f"║  Device:    {_device_name:<37s}║")
print(f"║  Mode:      {MODE:<37s}║")
print(f"╚══════════════════════════════════════════════════╝")

if MODE == "reduced-cpu":
    print("\n⚠️  No GPU detected. GPU-dependent cells will use CPU fallback.")
    print("   For full experience: Runtime → Change runtime type → T4 GPU")


---
## Part 0: Setup

In [ ]:
!pip install -q transformers==4.44.2

import os
import time
import torch
import torch.nn as nn
import torch._dynamo as dynamo

assert torch.cuda.is_available(), "GPU runtime required. Runtime → Change runtime type → GPU."
device = torch.device("cuda")
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")

torch.manual_seed(42)

In [ ]:
# PyTorch version compatibility check
import torch
print(f'PyTorch version: {torch.__version__}')
assert int(torch.__version__.split('.')[0]) >= 2, (
    f'This lab requires PyTorch 2.x or later. Got {torch.__version__}. '
    f'Run: pip install --upgrade torch'
)
print('✓ PyTorch version compatible')


---
### Model Definition

We use a minimal transformer block that exercises the full compilation pipeline:
attention → layer norm → FFN → residual. Small enough to compile quickly on Colab.

In [ ]:
class MiniTransformerBlock(nn.Module):
    """A clean transformer block suitable for torch.compile."""
    def __init__(self, d_model=512, nhead=8, dim_ff=2048):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_ff),
            nn.GELU(),
            nn.Linear(dim_ff, d_model),
        )

    def forward(self, x):
        # Pre-norm architecture
        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x


model = MiniTransformerBlock(d_model=512, nhead=8, dim_ff=2048).cuda().eval()
x = torch.randn(4, 64, 512, device=device)  # batch=4, seq_len=64, d_model=512

# Verify eager forward works
with torch.no_grad():
    eager_out = model(x)
print(f"Model output shape: {eager_out.shape}")
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

---
## Part 1: Compile and Measure Speedup

Use `torch.compile` to compile the model, then measure the speedup over eager execution.

**Important**: The first call to a compiled model triggers compilation (slow). Subsequent calls
use the cached compiled code (fast). Always warm up before measuring.

### Measurement Best Practices

> **⚠️ Single-run timings can be misleading.** GPU performance varies due to thermal throttling,
> background processes, and memory state. Always use multiple measurements.

**Protocol for reliable benchmarks:**
1. **Warmup:** 3-5 iterations (discarded) to fill caches and trigger JIT compilation
2. **Repeat:** N ≥ 10 measurement iterations
3. **Report:** Median (robust to outliers) ± IQR or percentiles (P5, P50, P95)
4. **Synchronize:** Always call `torch.cuda.synchronize()` before timing GPU ops
5. **Isolate:** Use `torch.inference_mode()` or `torch.no_grad()` to avoid autograd overhead


**💡 Hint: `bench_robust` (Part 1)**

<details>
<summary>Click to reveal a hint</summary>

- **Warm-up:** call `fn()` `n_warmup` times and throw the results away, then `torch.cuda.synchronize()`.
- **Measure:** synchronize, `t0 = time.perf_counter()`, `fn()`, synchronize, append `(time.perf_counter() - t0) * 1000`.
- **Return:** a dict with `median_ms`, `mean_ms`, `std_ms`, `p5_ms`, `p95_ms`, `iqr_ms`, `n` — use `np.median` and `np.percentile(times, [5, 25, 75, 95])`.
- Synchronizing on *both* sides of the timed region is what stops CUDA's asynchrony from timing a kernel launch as ~0 ms.

</details>

In [ ]:
# TODO: Implement a robust benchmarking utility
# Requirements:
# 1. Accept a callable fn and run it n_warmup times (discard results)
# 2. Run fn n_measure times, recording wall-clock time for each
# 3. Return statistics: median, mean, std, P5, P95, IQR
# 4. Handle CUDA synchronization (torch.cuda.synchronize() before/after)
#
# Hint: Use time.perf_counter() for high-resolution timing
# Hint: np.percentile(times, [5, 50, 95]) gives you P5, median, P95

import time
import numpy as np

def bench_robust(fn, n_warmup=5, n_measure=20, sync_cuda=True):
    """Run fn() with warmup, return timing statistics in milliseconds."""
    # YOUR CODE HERE
    pass

# Test your implementation:
if torch.cuda.is_available():
    x = torch.randn(1024, 1024, device=DEVICE)
    stats = bench_robust(lambda: torch.mm(x, x))
    print(f"Matrix multiply (1024x1024) on {DEVICE}:")
    print(f"  Median: {stats['median_ms']:.3f} ms")
    print(f"  P5-P95: [{stats['p5_ms']:.3f}, {stats['p95_ms']:.3f}] ms")
    print(f"  IQR:    {stats['iqr_ms']:.3f} ms (over {stats['n']} runs)")
    assert 'median_ms' in stats and stats['n'] == 20
else:
    print("Skipping bench_robust demo (no GPU)")


In [ ]:
# TODO: Compile the model with torch.compile (default settings)
x = torch.randn(4, 64, 512, device=device)  # same shape as model expects

compiled_model = None  # TODO: torch.compile(model)

# Warm up compiled model (triggers compilation)
print("Compiling (first call triggers JIT)...")
with torch.no_grad():
    _ = compiled_model(x)  # compilation happens here
    torch.cuda.synchronize()
print("Compilation complete.")

def measure_latency(fn, input_tensor, n_runs=50, warmup=10):
    """Measure average latency in ms. PROVIDED - do not edit."""
    with torch.no_grad():
        for _ in range(warmup):
            fn(input_tensor)
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(n_runs):
            fn(input_tensor)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
    return (elapsed / n_runs) * 1000  # ms

# TODO: measure both, then compute the speedup
eager_ms    = None  # TODO: measure_latency(model, x)
compiled_ms = None  # TODO: measure_latency(compiled_model, x)
speedup     = None  # TODO: eager_ms / compiled_ms

print(f"\nEager latency:    {eager_ms:.3f} ms")
print(f"Compiled latency: {compiled_ms:.3f} ms")
print(f"Speedup:          {speedup:.2f}x")


In [ ]:
# --- Checkpoint 1 ---
assert compiled_model is not None, "Compile the model with torch.compile()"
assert compiled_ms > 0, "Compiled latency must be measured"
assert speedup > 0.8, f"Expected some speedup or at least parity, got {speedup:.2f}x"

# Verify correctness: compiled output should match eager output
with torch.no_grad():
    eager_out = model(x)
    compiled_out = compiled_model(x)
max_diff = (eager_out - compiled_out).abs().max().item()
assert max_diff < 1e-3, f"Compiled output diverges from eager by {max_diff:.6f}"

print(f"[OK] Checkpoint 1 passed!")
print(f"  Speedup: {speedup:.2f}x")
print(f"  Max numerical difference: {max_diff:.2e} (acceptable for FP32)")

### Compilation Latency vs. Steady-State Performance

`torch.compile` has a one-time cost: the first call triggers tracing, optimization, and code generation.
After that, subsequent calls run the optimized code at steady-state speed.

Three metrics matter for production decisions:
1. **First-call latency**: includes compilation (can be 10–60s for large models)
2. **Steady-state latency**: optimized execution (the payoff)
3. **Break-even count**: how many forward passes before compilation cost is recouped

In [ ]:
import time

# Measure first-call (compilation) latency
torch._dynamo.reset()
model_compile = torch.compile(model)

if torch.cuda.is_available():
    torch.cuda.synchronize()
start = time.perf_counter()
with torch.no_grad():
    _ = model_compile(x)
if torch.cuda.is_available():
    torch.cuda.synchronize()
first_call_ms = (time.perf_counter() - start) * 1000

# Measure steady-state latency (average of N runs after warmup)
N_WARMUP, N_MEASURE = 3, 20
for _ in range(N_WARMUP):
    with torch.no_grad():
        _ = model_compile(x)

if torch.cuda.is_available():
    torch.cuda.synchronize()
start = time.perf_counter()
for _ in range(N_MEASURE):
    with torch.no_grad():
        _ = model_compile(x)
if torch.cuda.is_available():
    torch.cuda.synchronize()
steady_state_ms = (time.perf_counter() - start) * 1000 / N_MEASURE

# Measure eager baseline for comparison
model.eval()
if torch.cuda.is_available():
    torch.cuda.synchronize()
start = time.perf_counter()
for _ in range(N_MEASURE):
    with torch.no_grad():
        _ = model(x)
if torch.cuda.is_available():
    torch.cuda.synchronize()
eager_baseline_ms = (time.perf_counter() - start) * 1000 / N_MEASURE

# Break-even calculation
compilation_overhead_ms = first_call_ms - steady_state_ms
savings_per_call_ms = eager_baseline_ms - steady_state_ms
break_even = int(compilation_overhead_ms / savings_per_call_ms) if savings_per_call_ms > 0 else float('inf')

print(f"First-call (includes compilation): {first_call_ms:>10.1f} ms")
print(f"Steady-state (compiled):           {steady_state_ms:>10.2f} ms")
print(f"Eager baseline:                    {eager_baseline_ms:>10.2f} ms")
print(f"Speedup (steady-state vs eager):   {eager_baseline_ms/steady_state_ms:>10.2f}\u00d7")
print(f"Compilation overhead:              {compilation_overhead_ms:>10.1f} ms")
print(f"Break-even iterations:             {break_even:>10d}")

---
## Part 2: Backend Ablation

PyTorch's compilation stack has multiple stages. By changing the `backend` argument,
you can stop at different levels and isolate which stage provides the speedup:

| Backend | What it tests |
|---------|---------------|
| `"eager"` | Only TorchDynamo graph capture (no optimization) |
| `"aot_eager"` | + AOTAutograd (backward graph generation) |
| `"inductor"` | Full stack: + TorchInductor (Triton codegen, operator fusion) |

If a bug appears with `"inductor"` but not `"aot_eager"`, the bug is in TorchInductor.

In [ ]:
# TODO: Compile with each backend and measure latency
# Reset dynamo between compilations to avoid measuring a cached graph.

backends = ["eager", "aot_eager", "inductor"]
results = {}

for backend in backends:
    dynamo.reset()  # Clear compilation cache

    # TODO: Compile with this backend
    compiled = None  # TODO: torch.compile(model, backend=backend)

    # Warm up
    with torch.no_grad():
        _ = compiled(x)
        torch.cuda.synchronize()

    # Measure
    latency = measure_latency(compiled, x)

    # Check correctness vs eager
    with torch.no_grad():
        test_out = compiled(x)
    diff = (eager_out - test_out).abs().max().item()

    results[backend] = {"latency_ms": latency, "max_diff": diff}
    print(f"Backend '{backend}': {latency:.3f} ms, max_diff={diff:.2e}")

print(f"\nInductor speedup over eager backend:  {results['eager']['latency_ms']/results['inductor']['latency_ms']:.2f}x")
print(f"AOT-eager speedup over eager backend: {results['eager']['latency_ms']/results['aot_eager']['latency_ms']:.2f}x")


In [ ]:
# --- Checkpoint 2 ---
assert len(results) == 3, "Must test all three backends"
assert all(r["max_diff"] < 1e-2 for r in results.values()), "All backends must produce correct output"
assert results["inductor"]["latency_ms"] <= results["eager"]["latency_ms"] * 1.5, \
    "Inductor should not be dramatically slower than eager"

print("[OK] Checkpoint 2 passed!")
print(f"  Key insight: the speedup comes from {'inductor (fusion/codegen)' if results['inductor']['latency_ms'] < results['aot_eager']['latency_ms'] else 'aot_eager (graph capture)'}")

---
## Part 3: Graph Break Detection

A **graph break** occurs when TorchDynamo encounters Python code it cannot capture
into the computation graph. The model still runs, but the broken subgraphs fall back
to eager execution, silently losing the compilation benefit.

Below is a model with **deliberate graph breaks**. Your job: detect and count them.

In [ ]:
class BrokenTransformerBlock(nn.Module):
    """Same as MiniTransformerBlock but with graph-breaking operations."""
    def __init__(self, d_model=512, nhead=8, dim_ff=2048):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_ff),
            nn.GELU(),
            nn.Linear(dim_ff, d_model),
        )
        self.call_count = 0

    def forward(self, x):
        # --- Graph Break 1: .item() forces CPU sync ---
        max_val = x.max().item()
        if max_val > 1000:  # never true, but dynamo can't prove it
            x = x * 0.5

        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        x = x + attn_out

        # --- Graph Break 2: print() is not traceable ---
        self.call_count += 1
        print(f"Forward call #{self.call_count}", end="\r")

        x = x + self.ffn(self.norm2(x))

        # --- Graph Break 3: numpy conversion ---
        norm_val = x.norm().cpu().numpy()
        _ = norm_val  # unused but still breaks the graph

        return x


broken_model = BrokenTransformerBlock().cuda().eval()

In [ ]:
# TODO: Use torch._dynamo.explain() to detect graph breaks
# It reports the number of breaks, the reason for each, and how many
# subgraphs the model was split into.

dynamo.reset()

# TODO: call dynamo.explain on the broken model.
#       Note the current API shape: explain(fn)(*args), not explain(fn, *args).
explanation = None  # TODO: dynamo.explain(broken_model)(x)

print(explanation)

# TODO: Extract the counts from the explanation object
n_graph_breaks = None  # TODO: explanation.graph_break_count
n_graphs       = None  # TODO: explanation.graph_count

print(f"\n--- Summary ---")
print(f"Graph breaks: {n_graph_breaks}")
print(f"Resulting subgraphs: {n_graphs}")
print("Each break = a CPU sync + Python fallback + re-entry into compiled code")


In [ ]:
# Measure the performance impact of graph breaks
dynamo.reset()
compiled_broken = torch.compile(broken_model)

# Warm up (suppress print output)
import io, contextlib
with torch.no_grad(), contextlib.redirect_stdout(io.StringIO()):
    for _ in range(5):
        compiled_broken(x)
    torch.cuda.synchronize()

# Measure
with contextlib.redirect_stdout(io.StringIO()):
    broken_ms = measure_latency(compiled_broken, x)

print(f"Clean model (compiled):  {compiled_ms:.3f} ms")
print(f"Broken model (compiled): {broken_ms:.3f} ms")
print(f"Slowdown from breaks:    {broken_ms/compiled_ms:.2f}x")

In [ ]:
# --- Checkpoint 3 ---
assert n_graph_breaks is not None, "Extract the graph break count from explanation"
assert n_graph_breaks >= 2, f"Expected at least 2 graph breaks, found {n_graph_breaks}"
assert n_graphs is not None, "Extract the graph count"
assert n_graphs > n_graph_breaks, "Number of graphs should exceed number of breaks (graphs = breaks + 1)"

print(f"[OK] Checkpoint 3 passed!")
print(f"  Detected {n_graph_breaks} graph breaks → {n_graphs} subgraphs")
print(f"  Performance penalty: {broken_ms/compiled_ms:.1f}x slower than clean compilation")

---
## Part 4: Fix Graph Breaks

Now fix `BrokenTransformerBlock` to eliminate all graph breaks.

**Rules for fixing**:
- Remove or replace `.item()` calls. Use tensor operations instead
- Remove `print()` statements (or use `torch._dynamo.config.suppress_errors`)
- Remove `.numpy()` conversions. Stay in tensor land
- Do NOT change the mathematical behavior of the model

**💡 Hint: fixing the three breaks (Part 4)**

<details>
<summary>Click to reveal a hint</summary>

Each break has a one-line fix, and none of them changes the model's math:

- **`x.max().item()` + `if max_val > 1000:`** — the branch is *dead code*: the max of a `randn` tensor is never above 1000. Delete both lines. Resist the urge to keep the branch with `torch.where`; that preserves a comparison you do not need.
- **`print(f"Forward call #...")`** — delete it, and the `self.call_count += 1` that feeds it. A mutating integer attribute also triggers recompiles.
- **`x.norm().cpu().numpy()`** — the value is unused (`_ = norm_val`). Delete it. Any `.cpu()` or `.numpy()` inside `forward()` forces a host sync.

What should remain is exactly `MiniTransformerBlock.forward`: norm → attention → residual → norm → FFN → residual.

</details>

In [ ]:
# TODO: Write a graph-break-free version of BrokenTransformerBlock.
#
# Rules:
#   - Remove or replace .item() calls. Use tensor operations instead.
#   - Remove print() statements.
#   - Remove .numpy() / .cpu() conversions. Stay in tensor land.
#   - Do NOT change the mathematical behavior of the model.

class FixedTransformerBlock(nn.Module):
    """Graph-break-free version of BrokenTransformerBlock."""
    def __init__(self, d_model=512, nhead=8, dim_ff=2048):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_ff),
            nn.GELU(),
            nn.Linear(dim_ff, d_model),
        )

    def forward(self, x):
        # YOUR CODE HERE
        # Same math as MiniTransformerBlock: pre-norm attention + FFN + residuals,
        # with none of the three graph-breaking operations.
        raise NotImplementedError


In [ ]:
# Validate the fix
#
# Measurement hygiene matters more here than anywhere else in the lab, because
# we are comparing against compiled_ms measured back in Part 1:
#   - dynamo.explain() leaves Dynamo instrumented, so we reset AFTER it and
#     before compiling for the timed run. Skipping that reset is how this cell
#     ends up reporting the fixed model as ~2x slower than the clean one.
#   - The broken model hit Dynamo's recompile limit (8) in Part 3. That is
#     per-code-object state, but reset clears it either way.
#   - We also RE-measure the clean model here, so the fixed-vs-clean comparison
#     is taken under the same conditions in the same cell. Part 1's compiled_ms
#     was measured many cells (and several compiles) ago.
dynamo.reset()

fixed_model = FixedTransformerBlock().cuda().eval()
# Copy weights from the original clean model so this is the same function.
fixed_model.load_state_dict(model.state_dict())

# Check: zero graph breaks. Do this FIRST, then reset before timing.
fixed_explanation = dynamo.explain(fixed_model)(x)
fixed_breaks = fixed_explanation.graph_break_count
print(f"Graph breaks in fixed model: {fixed_breaks}")

# Compile and measure the FIXED model.
dynamo.reset()
compiled_fixed = torch.compile(fixed_model)
with torch.no_grad():
    _ = compiled_fixed(x)          # trigger compilation
    torch.cuda.synchronize()
fixed_ms = measure_latency(compiled_fixed, x)

# Re-measure the CLEAN model under the same conditions, as the reference point
# for "did removing the breaks restore whole-graph performance?".
dynamo.reset()
_clean_again = torch.compile(model)      # `model` is the original clean block
with torch.no_grad():
    _ = _clean_again(x)                  # trigger compilation
    torch.cuda.synchronize()
clean_ms_recheck = measure_latency(_clean_again, x)

# Verify correctness
with torch.no_grad():
    fixed_out = compiled_fixed(x)
fix_diff = (eager_out - fixed_out).abs().max().item()

print(f"\nFixed model latency:      {fixed_ms:.3f} ms")
print(f"Clean model (re-measured): {clean_ms_recheck:.3f} ms")
print(f"Broken model (Part 3):     {broken_ms:.3f} ms")
print(f"Speedup vs broken:        {broken_ms/fixed_ms:.2f}x")
print(f"Fixed / clean ratio:      {fixed_ms/clean_ms_recheck:.2f}x  (want ~1.0)")
print(f"Max diff vs eager:        {fix_diff:.2e}")


In [ ]:
# --- Checkpoint 4 ---
assert fixed_breaks == 0, f"Fixed model still has {fixed_breaks} graph breaks. Keep fixing!"
assert fix_diff < 1e-3, f"Fixed model output diverges too much: {fix_diff:.6f}"
assert fixed_ms < broken_ms * 1.5, "Fixed model should be no slower than the broken one."

# The real question this part asks: did removing the breaks restore whole-graph
# performance? Compare against the clean model re-measured in the SAME cell, not
# against Part 1's number from many compiles ago.
_ratio = fixed_ms / clean_ms_recheck
assert 0.6 <= _ratio <= 1.6, (
    f"Fixed model is {_ratio:.2f}x the clean model's latency; expected roughly "
    f"parity since they are the same computation. If this is far off, a stale "
    f"dynamo state is likely: re-run this cell after a dynamo.reset().")

print(f"[OK] Checkpoint 4 passed!")
print(f"  Zero graph breaks achieved!")
print(f"  Fixed {fixed_ms:.3f} ms vs clean {clean_ms_recheck:.3f} ms "
      f"({_ratio:.2f}x) vs broken {broken_ms:.3f} ms")


---
## Part 5: Inspect Generated Code

`torch.compile` does not just run your model faster — it *rewrites* it. Set
`TORCH_COMPILE_DEBUG=1` and Inductor dumps the artifacts of that rewrite,
including `output_code.py`: the actual Python + Triton source it generated and
executed.

Two kinds of kernel appear in that file, and the split is the whole point of
this part:

| In `output_code.py` | What it is | Who runs it |
|---|---|---|
| `def triton_...` | A kernel Inductor **generated and fused** | Triton |
| `extern_kernels.mm` / `.addmm` / `.bmm` | A call out to a **hand-tuned library** | cuBLAS |

Inductor sends the GEMMs to cuBLAS (a tuned library beats generated code at
matmul) and fuses the surrounding elementwise and normalization work into a
handful of Triton kernels. Those fused kernels are the dataflow idea in
software: an intermediate stays in registers instead of making a round trip to
device memory.

Getting this dump to appear at all takes three things, and missing any one of
them makes the cell print `0 files` while *looking* like it worked:

1. **Set the config object, not the env var.** `TORCH_COMPILE_DEBUG` is read
   when `torch._inductor.config` is first imported. Torch was imported ~25 cells
   ago, so setting the variable here has no effect; we set
   `inductor_config.trace.enabled` directly.
2. **Defeat the code cache.** Inductor caches generated code on disk, and
   Parts 1–4 already compiled this model. A cache hit skips codegen entirely —
   it writes no artifacts yet still creates the directory and logs a
   `debug trace: …` path, which is exactly the misleading state this lab hit.
   `force_disable_caches = True` forces a cold compile.
3. **`dynamo.reset()` first**, so Dynamo re-traces rather than reusing its
   in-process compiled wrapper.

In [ ]:
# 5.1: dump Inductor's compilation artifacts to a known-clean directory.
#
# The subtlety that makes or breaks this cell: TORCH_COMPILE_DEBUG is read from
# the environment *when torch._inductor.config is first imported*. By the time
# this notebook runs, torch is long since imported, so setting the env var here
# has NO effect -- which is why an earlier version of this lab reported
# "0 files" and silently fell back to a stale cache fragment.
#
# The in-session switch is the config object itself. We set both: the config
# flags (what actually works now) and the env vars (so a fresh subprocess or a
# re-import would also dump).
import os, glob, shutil
from torch._inductor import config as inductor_config

DEBUG_DIR = "/tmp/torch_compile_debug_lab3"
shutil.rmtree(DEBUG_DIR, ignore_errors=True)
os.makedirs(DEBUG_DIR, exist_ok=True)

os.environ["TORCH_COMPILE_DEBUG"]     = "1"
os.environ["TORCH_COMPILE_DEBUG_DIR"] = DEBUG_DIR

# The flags that take effect in an already-running session.
_prev = (inductor_config.trace.enabled,
         inductor_config.trace.debug_dir,
         getattr(inductor_config, "force_disable_caches", False))
inductor_config.trace.enabled     = True
inductor_config.trace.debug_dir   = DEBUG_DIR
inductor_config.trace.output_code = True        # the file we want
inductor_config.trace.fx_graph    = True

# The second half of the problem, and the one that actually caused this cell to
# report "0 files": Inductor caches generated code on disk. Parts 1-4 already
# compiled this very model, so this compile would be a CACHE HIT -- it skips
# codegen, writes no artifacts, and still creates the directory and logs a
# "debug trace: ..." path. Forcing a cold compile is what makes the dump real.
inductor_config.force_disable_caches = True

# dynamo.reset() FIRST: a cached compilation emits no artifacts at all, so
# without this the cell can "succeed" and write nothing.
dynamo.reset()
debug_model = torch.compile(MiniTransformerBlock().cuda().eval(), backend="inductor")
try:
    with torch.no_grad():
        _ = debug_model(x)
        torch.cuda.synchronize()
finally:
    # Always restore, even if the compile raised: leaving tracing on makes
    # every later compile in the session slow and noisy.
    (inductor_config.trace.enabled,
     inductor_config.trace.debug_dir,
     inductor_config.force_disable_caches) = _prev
    os.environ.pop("TORCH_COMPILE_DEBUG", None)
    os.environ.pop("TORCH_COMPILE_DEBUG_DIR", None)

# Inductor writes its artifacts DURING compilation and logs the directory on
# the way out ("... debug trace: /path"), so by now everything exists. We walk
# the tree rather than trusting a single glob, and report directories too: an
# empty <name>.N directory means tracing was enabled but the individual dumps
# (trace.output_code etc.) were off or the graph was served from cache.
_all_files, _all_dirs = [], []
for _root, _dirnames, _filenames in os.walk(DEBUG_DIR):
    for _d in _dirnames:
        _all_dirs.append(os.path.join(_root, _d))
    for _f in _filenames:
        _all_files.append(os.path.join(_root, _f))
artifacts = sorted(_all_files)

print(f"Debug artifacts produced: {len(artifacts)} files "
      f"in {len(_all_dirs)} directories under {DEBUG_DIR}")
for _a in artifacts[:25]:
    print(f"  .{_a[len(DEBUG_DIR):]}  ({os.path.getsize(_a)} bytes)")
if len(artifacts) > 25:
    print(f"  ... and {len(artifacts) - 25} more")

if not artifacts:
    # Show the empty skeleton: it tells us tracing ran but dumped nothing.
    for _d in sorted(_all_dirs)[:10]:
        print(f"  [dir] .{_d[len(DEBUG_DIR):]}  (empty)")
    print("\n[warn] The debug directory exists but holds no files, which means")
    print("       Inductor served this graph from its code cache and skipped")
    print("       codegen entirely. Force a cold compile and re-run:")
    print("         import torch._inductor.config as ic; ic.force_disable_caches = True")
    print("       Cell 5.2 falls back to the code cache meanwhile; treat its")
    print("       counts with suspicion.")


In [ ]:
# 5.2: find output_code.py, count fused Triton kernels vs extern cuBLAS calls.
#
# The counts are the gradeable evidence from Part 5, so we fall back through
# several ways of getting the generated source rather than giving up:
#   1. output_code.py from the TORCH_COMPILE_DEBUG dump (the normal path)
#   2. any *.py in the Inductor codegen cache that defines a triton kernel
# If neither works the variables stay None and Part 7 exports nulls, which the
# grader notebook treats as "not collected" rather than as a wrong answer.
import re

def count_kernels(code_text):
    """(triton_kernels, extern_kernels) in a block of generated Inductor code.

    "extern" means "this matmul was handed to a tuned library instead of being
    generated". Inductor spells that several ways depending on version and
    whether max-autotune picked a Triton GEMM instead, so match the family:
      extern_kernels.mm / .addmm / .bmm      (classic wrapper)
      aten.mm / aten.addmm / aten.bmm        (direct aten dispatch)
      torch.ops.aten.*                       (fully qualified form)
    A build that generates its GEMMs in Triton legitimately reports 0 here.
    """
    # Count DEFINITIONS of generated kernels, not call sites, so a kernel
    # invoked in a loop is still one kernel.
    triton = len(re.findall(r"^\s*def\s+triton_", code_text, re.M))
    if triton == 0:                      # newer codegen names them via a decorator
        triton = len(re.findall(r"@triton\.jit", code_text))
    extern = len(re.findall(
        r"extern_kernels\.\w+"
        r"|(?:torch\.ops\.)?aten\.(?:mm|addmm|bmm|_scaled_mm)\b",
        code_text))
    return triton, extern

output_code_path = None
output_code      = None
codegen_source   = None      # "debug-dump" | "code-cache" | None

_candidates = sorted(glob.glob(f"{DEBUG_DIR}/**/output_code.py", recursive=True))
if _candidates:
    # The normal path: one output_code.py per compiled graph. Concatenate them
    # so the counts cover the whole block, not just the first graph.
    output_code_path = _candidates[0] if len(_candidates) == 1 else \
                       f"{len(_candidates)} output_code.py files"
    output_code      = "\n".join(open(_p).read() for _p in _candidates)
    codegen_source   = "debug-dump"
else:
    # Fallback: Inductor's on-disk code cache. Each generated kernel lands in
    # its own file, so take every file written since this cell's compile and
    # concatenate - reading a single file reports a FRAGMENT and undercounts.
    # Wrapped broadly on purpose: a torch build that moves codecache must not
    # take the whole cell down with it.
    try:
        from torch._inductor import codecache
        _cache = getattr(codecache, "cache_dir", lambda: "/tmp/torchinductor_root")()
        _recent = sorted(glob.glob(f"{_cache}/**/*.py", recursive=True),
                         key=os.path.getmtime, reverse=True)[:40]
        _parts = []
        for _p in _recent:
            try:
                _t = open(_p).read()
            except OSError:
                continue
            if "triton_" in _t or "extern_kernels." in _t:
                _parts.append(_t)
        if _parts:
            output_code_path = f"{len(_parts)} file(s) in {_cache}"
            output_code      = "\n".join(_parts)
            codegen_source   = "code-cache"
    except Exception as _e:
        print(f"[note] code-cache fallback unavailable ({type(_e).__name__}).")

if output_code is None:
    triton_kernels = extern_kernels = None
    print("Could not locate generated code. triton/extern counts will export as null.")
    print("Re-run cell 5.1 after a dynamo.reset() if you want the counts.")
else:
    triton_kernels, extern_kernels = count_kernels(output_code)
    _total = triton_kernels + extern_kernels
    print(f"Source: {codegen_source}")
    if codegen_source == "code-cache":
        print("  [warn] read from Inductor's code cache, not the debug dump.")
        print("         These counts may be INCOMPLETE - the cache stores each")
        print("         kernel separately and unrelated compiles can mix in.")
    print(f"Generated code: {output_code_path}")
    print(f"  ({len(output_code.splitlines())} lines)\n")
    print("--- Kernel statistics ---")
    print(f"  Triton (fused) kernels : {triton_kernels}")
    print(f"  extern (cuBLAS) calls  : {extern_kernels}")
    print(f"  Total launched kernels : {_total}")
    if _total:
        print(f"  Fusion ratio           : {triton_kernels}/{_total} "
              f"= {triton_kernels/_total:.0%} of kernels are generated/fused")
    # Show which call forms actually matched, so a zero count is diagnosable
    # from the output instead of being a mystery.
    _names = re.findall(r"^\s*def\s+(triton_\w+)", output_code, re.M)
    _calls = re.findall(r"extern_kernels\.\w+"
                        r"|(?:torch\.ops\.)?aten\.(?:mm|addmm|bmm|_scaled_mm)\b",
                        output_code)
    if _names:
        print("\n  generated kernels: " + ", ".join(sorted(set(_names))[:8]))
    if _calls:
        from collections import Counter
        print("  library calls:     " + ", ".join(
            f"{_k} x{_v}" for _k, _v in Counter(_calls).most_common(8)))
    else:
        print("  library calls:     none found (matmuls generated in Triton)")

    print("\n--- First 60 lines of the generated code ---")
    print("\n".join(output_code.splitlines()[:60]))


In [ ]:
# --- Checkpoint 5 -------------------------------------------------------
# Part 5's gradeable evidence is the kernel split, so check it here. Two things
# are deliberately NOT asserted, because both vary by torch version and would
# fail honest runs:
#   - that any extern (cuBLAS) call appears. Newer Inductor builds can generate
#     the GEMMs in Triton instead of calling out to a library.
#   - that fusion reduced the launch count below the ~8 logical ops.
# The counts may also be None on builds that refuse to dump artifacts.
if triton_kernels is None or extern_kernels is None:
    print("[warn] kernel counts unavailable; Part 7 will export null for both.")
    print("       This does not cost you points, but you miss the Part 5 lesson:")
    print("       re-run 5.1 and 5.2 if you can.")
else:
    assert triton_kernels >= 0 and extern_kernels >= 0, "Kernel counts cannot be negative."
    _total = triton_kernels + extern_kernels
    assert _total >= 1, (
        "A compiled transformer block must launch at least one kernel. "
        "Did cell 5.2 parse the right file?")
    print(f"[OK] Checkpoint 5 passed!")
    print(f"  {triton_kernels} generated Triton kernel(s) + {extern_kernels} "
          f"library (cuBLAS) call(s) = {_total} launches")
    print(f"  The block has ~8 logical PyTorch ops.")
    if extern_kernels == 0:
        print("  No library calls found: this build generated its matmuls in")
        print("  Triton rather than dispatching to cuBLAS. That is a real and")
        print("  interesting result - note it for the Part 5 discussion.")
    else:
        print(f"  The matmuls went to cuBLAS; the norms, GELU, bias and residual")
        print(f"  adds were fused into the generated kernels.")


### Check your understanding (ungraded)

These prompts are **not graded and not submitted** — nothing here travels in
your JSON, and no one reviews your answers. Work through them anyway: this is
the reasoning the **Concept Check quiz** tests, and that quiz *is* the graded
one, worth **10 graded points**. Compare your answers against the solution
notebook once it unlocks.

---

**1.** Your compile speedup was probably small, and on some torch versions
`inductor` lands *slower* than the `eager` backend. Explain why, in terms of
what fusion actually buys you. What would you change about the workload to make
compilation pay off?

*Your reasoning here (ungraded — for your own learning).*

---

**2.** Look at your `compilation_overhead_ms` and `savings_per_call_ms` from
Part 1. What does their ratio imply about *when* you should compile at all? Name
a deployment where compiling is clearly right and one where it is clearly wrong.

*Your reasoning here (ungraded — for your own learning).*

---

**3.** Why does `.item()` force a graph break? Why is that a *performance*
problem on a GPU but potentially a *correctness* problem for a dataflow
accelerator's compiler?

*Your reasoning here (ungraded — for your own learning).*

---

**4.** You measured some number of graph breaks and one more subgraph than
breaks. Why is that relation exact rather than approximate?

*Your reasoning here (ungraded — for your own learning).*

---

**5.** In Part 5 you counted generated Triton kernels and calls out to cuBLAS.
Inductor sent the matmuls to the library rather than generating them. Was that
the compiler giving up? What *is* the compiler's contribution here?

*Your reasoning here (ungraded — for your own learning).*

In [ ]:
# --- Lab 3 summary -----------------------------------------------------
print("=" * 60)
print("LAB 3 COMPLETE")
print("=" * 60)
print("Summary of what you measured:")
print(f"  Part 1 - compile speedup        : {speedup:.2f}x "
      f"({eager_ms:.3f} ms eager -> {compiled_ms:.3f} ms compiled)")
print(f"  Part 1 - compilation overhead   : {compilation_overhead_ms:.0f} ms "
      f"(break-even at {break_even:,} iterations)" if savings_per_call_ms > 0 else
      f"  Part 1 - compilation overhead   : {compilation_overhead_ms:.0f} ms "
      f"(never breaks even: compiled is not faster here)")
_fastest = min(results, key=lambda b: results[b]["latency_ms"])
print(f"  Part 2 - fastest backend        : {_fastest} "
      f"({results[_fastest]['latency_ms']:.3f} ms)")
print(f"  Part 3 - graph breaks detected  : {n_graph_breaks} -> {n_graphs} subgraphs")
print(f"  Part 4 - breaks after the fix   : {fixed_breaks}")
if triton_kernels is not None:
    print(f"  Part 5 - kernel split           : {triton_kernels} fused Triton "
          f"+ {extern_kernels} cuBLAS")
else:
    print(f"  Part 5 - kernel split           : not collected")
print("=" * 60)


---

## Part 6 - Update the Accelerator Landscape Map

Add this week's row to the shared map you have been building since Lab 1.


In [ ]:
# Part 6 -- Landscape Map row for Week 3. Fill the measured values from your run.
landscape_row_w3 = {
    "accelerator":               (f"GPU ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "CPU"),
    "graph_capture":             "TorchDynamo (torch.compile)",
    "lowering_pipeline":         "Dynamo -> AOTAutograd -> TorchInductor",
    "codegen":                   (f"Triton ({triton_kernels} fused) + "
                                  f"extern/cuBLAS ({extern_kernels} calls)"
                                  if triton_kernels is not None else
                                  "TorchInductor: Triton codegen + extern library calls"),
    "eager_vs_compiled_speedup": None,   # TODO: round(speedup, 2)
    "graph_breaks_before_fix":   None,   # TODO: int(n_graph_breaks)
    "graph_breaks_after_fix":    None,   # TODO: int(fixed_breaks)
}
for _k, _v in landscape_row_w3.items():
    print(f"  {_k:27s}: {_v}")


In [ ]:
# --- Checkpoint 6 ------------------------------------------------------
required = {"accelerator", "graph_capture", "lowering_pipeline", "codegen",
            "eager_vs_compiled_speedup", "graph_breaks_before_fix",
            "graph_breaks_after_fix"}
assert set(landscape_row_w3.keys()) == required, \
    f"landscape_row_w3 keys are wrong: {sorted(set(landscape_row_w3) ^ required)}"
assert all(v is not None for v in landscape_row_w3.values()), \
    "every landscape_row_w3 field must be filled in"
# The measured fields must mirror what you actually measured.
assert abs(float(landscape_row_w3["eager_vs_compiled_speedup"]) - speedup) < 0.02, \
    "eager_vs_compiled_speedup should be round(speedup, 2) from Part 1"
assert int(landscape_row_w3["graph_breaks_before_fix"]) == int(n_graph_breaks), \
    "graph_breaks_before_fix should be n_graph_breaks from Part 3"
assert int(landscape_row_w3["graph_breaks_after_fix"]) == int(fixed_breaks), \
    "graph_breaks_after_fix should be fixed_breaks from Part 4"
print("[OK] Checkpoint 6 passed!")
print(f"  Week 3 Landscape Map row complete: speedup "
      f"{landscape_row_w3['eager_vs_compiled_speedup']}x, breaks "
      f"{landscape_row_w3['graph_breaks_before_fix']} -> "
      f"{landscape_row_w3['graph_breaks_after_fix']}")


---
## Part 7: Save and export your results

Run the cell below. It writes **`lab3_results.json`** and prints it inline so
you can copy it without opening the file browser.

Then open the **grader notebook** on Coursera Labs, paste the JSON into its
`RESULTS_JSON` cell, and run every cell to see your **40 ungraded
Self-Assessment points**.

Everything it checks comes from values you measured in Parts 1-6: the eager and
compiled latencies, the compilation overhead, the three backend latencies, the
graph-break counts before and after your fix, the kernel split from the
generated code, and your Week-3 landscape row.

> Those 40 points are **feedback, not a grade** — they tell you how well the lab
> went. The graded item for this week is the **Concept Check quiz** (10 points,
> auto-graded on Coursera, 80% to pass).

> **Your absolute timings are never compared against a reference.** The grader
> notebook checks that your numbers are **consistent with each other**, not that
> they are fast.

In [ ]:
# === Part 7: Save your results for self-assessment =====================
# This writes lab3_results.json. You will paste its contents into the GRADER
# notebook on Coursera Labs, which reports the 40 ungraded Self-Assessment
# points. Without it there is nothing to check.
#
# PROVIDED - do not edit. Everything here is built from variables you already
# computed in Parts 1-6, so there is nothing to fill in and nothing to get
# wrong. Just run it.
#
# Everything the grader notebook needs is here. It exports the RAW latencies
# alongside the derived ratios, so the checks can recompute every speedup from
# your own measurements rather than trusting the ratio you typed.
import json as _json

RESULTS = {
    # --- Part 1: compile and measure -------------------------------------
    "perf": {
        "eager_ms":    eager_ms,
        "compiled_ms": compiled_ms,
        "speedup":     speedup,
        "max_diff":    max_diff,
    },
    # --- Part 1b: compilation overhead vs steady state -------------------
    # break_even is a ratio of two DIFFERENCES, so it is extremely sensitive to
    # rounding. Export the raw inputs too; the grader recomputes from these.
    "compile_cost": {
        "first_call_ms":           first_call_ms,
        "steady_state_ms":         steady_state_ms,
        "eager_baseline_ms":       eager_baseline_ms,
        "compilation_overhead_ms": compilation_overhead_ms,
        "savings_per_call_ms":     savings_per_call_ms,
        # break_even can be math.inf when compilation never pays off. JSON has
        # no infinity, so export None and let the grader read it as "never".
        "break_even_iters":        (int(break_even)
                                    if break_even != float("inf") else None),
    },
    # --- Part 2: backend ablation (raw per-backend latencies) ------------
    "backends": {
        _b: {
            "latency_ms": float(_v["latency_ms"]),
            "max_diff":   float(_v["max_diff"]),
        }
        for _b, _v in results.items()
    },
    # --- Part 3: graph breaks -------------------------------------------
    "graph_breaks": {
        "n_graph_breaks": int(n_graph_breaks),
        "n_graphs":       int(n_graphs),
        "broken_ms":      broken_ms,
        "clean_ms":       compiled_ms,
        "slowdown":       broken_ms / compiled_ms,
    },
    # --- Part 4: the fix ------------------------------------------------
    "graph_break_fix": {
        "fixed_breaks":      int(fixed_breaks),
        "fixed_ms":          fixed_ms,
        "fix_diff":          fix_diff,
        "speedup_vs_broken": broken_ms / fixed_ms,
        # The clean model re-measured in the SAME cell as fixed_ms. The grader
        # compares fixed_ms against THIS rather than against perf.compiled_ms,
        # which was measured many compiles earlier and drifts on a contended VM.
        "clean_ms_recheck":  clean_ms_recheck,
    },
    # --- Part 5: the kernel split from the generated code ---------------
    # Counts, not prose. null is accepted when the dump could not be parsed.
    "codegen": {
        "triton_kernels": (int(triton_kernels)
                           if triton_kernels is not None else None),
        "extern_kernels": (int(extern_kernels)
                           if extern_kernels is not None else None),
        # Where the counts came from: "debug-dump" is authoritative,
        # "code-cache" is a best-effort fallback that may undercount, null
        # means nothing was parsed. The grader accepts all three.
        "source": codegen_source,
    },
    # --- Part 6: landscape map row --------------------------------------
    "landscape_row_w3": landscape_row_w3,
    # --- environment (reported, never checked against) ------------------
    "environment": {
        "device": (torch.cuda.get_device_name(0)
                   if torch.cuda.is_available() else "CPU only"),
        "torch_version": torch.__version__,
        "mode": MODE,
    },
}

with open("lab3_results.json", "w") as _f:
    _json.dump(RESULTS, _f, indent=2)

print("=" * 70)
print("lab3_results.json  --  copy everything between the lines below")
print("=" * 70)
print(_json.dumps(RESULTS, indent=2))
print("=" * 70)
print("\nNEXT STEPS")
print("-" * 70)
print("1. Copy the JSON printed above (from the opening '{' through the")
print("   final '}'), or open the file browser (folder icon, left sidebar)")
print("   and double-click lab3_results.json.")
print("2. Open the GRADER notebook on Coursera Labs (self-assessment).")
print("3. Paste the contents into the RESULTS_JSON cell, then Run All.")
print("   It reports your 40 ungraded Self-Assessment points as feedback.")
print("4. The Concept Check quiz is the GRADED item: a separate auto-graded")
print("   Coursera Assignment worth 10 points (80% to pass). Take it as well.")
print("\nKeep this Colab notebook: if a self-assessment check fails you will")
print("want to re-run the cell that produced the value.")

# Optional: download the file instead of copy-pasting.
# from google.colab import files
# files.download('lab3_results.json')


---
## Optional: Dataflow Deep Dive (Spyre hardware access required)

> *Only complete this section if you have been granted Spyre access through the IBM Research academic partnership.*

1. Compile the same `MiniTransformerBlock` for Spyre using `torch.compile(model, backend="sendnn")`
2. Set `TORCH_COMPILE_DEBUG=1` and examine the `sdsc_*.json` files (Spyre's backend IR)
3. Compare: how does Spyre's compilation artifact structure differ from Inductor's Triton output?
4. Identify which operators Spyre fuses vs. which it keeps as separate kernels

---

*Course: Dataflow Architectures for AI, Week 3 Lab, v1.0 · © 2026 Dr. Kaoutar El Maghraoui*

> **Spyre vocabulary (consistent across all labs):** `import torch_sendnn` registers the `aiu` device; compile with `torch.compile(model, backend="sendnn")`; dump the backend IR with `TORCH_COMPILE_DEBUG=1` (look for `sdsc_*.json`); raise verbosity with `SENDNN_LOG_LEVEL=DEBUG`. Requires Spyre access via an IBM Research partnership.
